In [1]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import json

In [2]:
def load_api_key():
    """
    Load Steam API keys from the .env file located in the .venv folder.
    
    Returns:
        tuple: A tuple containing (api_key).
    """
    # Notebook is in the 'src/' folder, so go up one level to reach '.venv/.env'
    env_path = r'..\src\config.env'
    
    load_dotenv(dotenv_path=env_path)
    api_key = os.getenv('itd_api_key')
    return api_key

api_key = load_api_key()


In [5]:
games = pd.read_json(r'..\data\games_id_all.json')
games = games.loc['apps', 'response']

def name_formatting(id):
    game_data = games[id]
    #print(games.loc['apps', 'response'][id])
    game_name = game_data['name']
    game_id = game_data['appid']
    formatted_name = game_name.lower().replace(' ', '-').replace(':', '')
    return formatted_name, game_id

In [12]:
for index, game in enumerate(games):

    if(index == 1):          #temporary stop
        break

    URL = 'https://api.isthereanydeal.com/games/search/v1'

    game_name, game_id = name_formatting(index)
    new_entry = {"appid": game_id, "name": game_name}

    params = {
        'key': api_key,
        'title': game_name         #fetching game name to get uuid
    }

    response = requests.get(URL, params=params)

    if response.status_code == 200:
        data = response.json()

        game_id = data[0]["id"]

        URL_price = 'https://api.isthereanydeal.com/games/history/v2'       #fetching  price data

        params = {
            'key': api_key,
            'id': game_id,
            'shops': 61,
            'country': 'PL'
            }

        response = requests.get(URL_price, params=params)

        if response.status_code == 200:
            data = response.json()
            data.insert(0, new_entry)
            print(data)
        else:
            print(f"Error during prices fetching: {response.status_code}")
    else:
        print(f"Error during id fetching: {response.status_code}")

[{'appid': 10, 'name': 'counter-strike'}, {'timestamp': '2026-03-26T18:20:05+01:00', 'shop': {'id': 61, 'name': 'Steam'}, 'deal': {'price': {'amount': 45.99, 'amountInt': 4599, 'currency': 'PLN'}, 'regular': {'amount': 45.99, 'amountInt': 4599, 'currency': 'PLN'}, 'cut': 0}}, {'timestamp': '2026-03-19T19:18:51+01:00', 'shop': {'id': 61, 'name': 'Steam'}, 'deal': {'price': {'amount': 9.19, 'amountInt': 919, 'currency': 'PLN'}, 'regular': {'amount': 45.99, 'amountInt': 4599, 'currency': 'PLN'}, 'cut': 80}}, {'timestamp': '2026-02-03T18:46:28+01:00', 'shop': {'id': 61, 'name': 'Steam'}, 'deal': {'price': {'amount': 45.99, 'amountInt': 4599, 'currency': 'PLN'}, 'regular': {'amount': 45.99, 'amountInt': 4599, 'currency': 'PLN'}, 'cut': 0}}]
